# Paper Pipeline (CGA-CPSO-CSVM):
  1. Data description: combined UCI heart-disease dataset (918 unique rows,
     11 features), 80:20 train/test split -> 734 train / 184 test.
  2. CGA: genetic-algorithm wrapper feature selection (roulette-wheel
     selection, one-point crossover, bit-flip mutation) -> paper reports
     9 of 11 features selected (~19% reduction).
  3. CPSO: particle swarm optimization of the SVM hyperparameters (C, gamma).
  4. CSVM: final RBF-kernel SVM trained on the selected features and tuned
     hyperparameters, evaluated with Accuracy, Precision, Sensitivity
     (Recall), Specificity, F1, MCC, ROC-AUC, LR+, LR-, and DOR -- the same
     metric set reported in the paper's "Performance metrics" section.
  5. Tenfold cross-validation of the final model, as described in the
     paper's "K-fold cross-validation" section.

Importing Data

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
    matthews_corrcoef, roc_auc_score, roc_curve,
)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

In [14]:
# importing Dataset

heart_disease = pd.read_csv("UCI-918-11.csv")
# Verifying if it loaded correctly
heart_disease.shape

(918, 12)

In [22]:
def prepare_data(df, target_col="target"):
    feature_names = [c for c in df.columns if c != target_col]
    corr_with_target = df[feature_names + [target_col]].corr(numeric_only=True)[target_col].drop(target_col)
    X = df[feature_names].values.astype(float)
    y = df[target_col].values.astype(int)
    return X, y, feature_names, corr_with_target

# Fitness Helper:
CSVM trained on feature subset then evaluated with stratified k-fold cross validation accuracy,

In [23]:
def evaluate_csvm(X, y, feature_mask=None, C=1.0, gamma="scale", cv_splits=5):
    if feature_mask is not None:
        if feature_mask.sum() == 0:
            return 0.0
        X_subset = X[:, feature_mask.astype(bool)]
    else:
        X_subset = X

    clf = SVC(C=C, gamma=gamma, kernel="rbf", random_state=RANDOM_STATE)
    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    return cross_val_score(clf, X_subset, y, cv=skf, scoring="accuracy").mean()

 # Feature selection using CGA (classical genetic algorithm)
population, fitness evaluation, roulette-wheel selection, one-pointcrossover, bit-flip mutation, generational replacement, convergence.

In [24]:
def cga_feature_selection(
    X, y,
    population_size=20,
    generations=30,
    crossover_rate=0.8,
    mutation_rate=0.05,
    patience=6,
    verbose=True,
):
    n_features = X.shape[1]

    # Initialize population P with N chromosomes (binary encoded feature subsets)
    population = rng.integers(0, 2, size=(population_size, n_features))
    for i in range(population_size):
        if population[i].sum() == 0:
            population[i, rng.integers(n_features)] = 1

    best_fitness_history = []
    best_chromosome = None
    best_fitness = -np.inf
    stale = 0

    for gen in range(1, generations + 1):
        # Evaluate fitness of each chromosome (train CSVM -> CV accuracy)
        fitness = np.array([evaluate_csvm(X, y, chromo) for chromo in population])

        gen_best_idx = np.argmax(fitness)
        if fitness[gen_best_idx] > best_fitness:
            best_fitness = fitness[gen_best_idx]
            best_chromosome = population[gen_best_idx].copy()
            stale = 0
        else:
            stale += 1

        best_fitness_history.append(best_fitness)
        if verbose:
            print(f"[CGA] Gen {gen:02d} | best fitness = {best_fitness:.4f} "
                  f"| features = {int(best_chromosome.sum())}/{n_features}")

        if stale >= patience:
            if verbose:
                print(f"[CGA] Converged after {gen} generations.")
            break

        # Roulette wheel selection
        shifted = fitness - fitness.min() + 1e-6
        probs = shifted / shifted.sum()
        parents = population[rng.choice(population_size, size=population_size, p=probs)]

        # One-point crossover
        offspring = parents.copy()
        for i in range(0, population_size - 1, 2):
            if rng.random() < crossover_rate:
                point = rng.integers(1, n_features)
                offspring[i, point:], offspring[i + 1, point:] = (
                    offspring[i + 1, point:].copy(), offspring[i, point:].copy(),
                )

        # Bit-flip mutation
        flip_mask = rng.random((population_size, n_features)) < mutation_rate
        offspring[flip_mask] = 1 - offspring[flip_mask]
        for i in range(population_size):
            if offspring[i].sum() == 0:
                offspring[i, rng.integers(n_features)] = 1

        # Replace least-fit chromosomes with offspring
        n_replace = population_size // 2
        worst_idx = np.argsort(fitness)[:n_replace]
        population[worst_idx] = offspring[:n_replace]

    return best_chromosome.astype(bool), best_fitness, best_fitness_history

# Hyperparameter tuning using CPSO (classical particle swarm optimization)

In [25]:
def cpso_hyperparameter_tuning(
    X, y, best_features,
    swarm_size=15,
    iterations=25,
    w=0.7, c1=1.5, c2=1.5,
    bounds=((0.01, 100.0), (0.0001, 10.0)),  # (C range, gamma range)
    patience=6,
    verbose=True,
):
    (c_low, c_high), (g_low, g_high) = bounds
    X_sel = X[:, best_features]

    pos = np.column_stack([
        rng.uniform(c_low, c_high, swarm_size),
        rng.uniform(g_low, g_high, swarm_size),
    ])
    vel = np.zeros((swarm_size, 2))

    pbest_pos = pos.copy()
    pbest_fit = np.array([evaluate_csvm(X_sel, y, None, C=p[0], gamma=p[1]) for p in pos])
    gbest_idx = np.argmax(pbest_fit)
    gbest_pos, gbest_fit = pbest_pos[gbest_idx].copy(), pbest_fit[gbest_idx]

    history = [gbest_fit]
    stale = 0

    for it in range(1, iterations + 1):
        fitness = np.array([evaluate_csvm(X_sel, y, None, C=p[0], gamma=p[1]) for p in pos])

        improved = fitness > pbest_fit
        pbest_fit[improved] = fitness[improved]
        pbest_pos[improved] = pos[improved]

        it_best_idx = np.argmax(pbest_fit)
        if pbest_fit[it_best_idx] > gbest_fit:
            gbest_fit = pbest_fit[it_best_idx]
            gbest_pos = pbest_pos[it_best_idx].copy()
            stale = 0
        else:
            stale += 1

        history.append(gbest_fit)
        if verbose:
            print(f"[CPSO] Iter {it:02d} | gBest fitness = {gbest_fit:.4f} "
                  f"| C={gbest_pos[0]:.4f}, gamma={gbest_pos[1]:.4f}")

        if stale >= patience:
            if verbose:
                print(f"[CPSO] Converged after {it} iterations.")
            break

        r1 = rng.random(swarm_size)
        r2 = rng.random(swarm_size)
        for dim in range(2):
            vel[:, dim] = (
                w * vel[:, dim]
                + c1 * r1 * (pbest_pos[:, dim] - pos[:, dim])
                + c2 * r2 * (gbest_pos[dim] - pos[:, dim])
            )
        pos = pos + vel
        pos[:, 0] = np.clip(pos[:, 0], c_low, c_high)
        pos[:, 1] = np.clip(pos[:, 1], g_low, g_high)

    return {"C": gbest_pos[0], "gamma": gbest_pos[1]}, gbest_fit, history

# Metrics:
 Accuracy, Precision, Sensitivity (Recall), Specificity, F1, MCC, AUC, LR+, LR-, DOR

In [26]:
def compute_full_metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = recall_score(y_true, y_pred)          # a.k.a. recall / TPR
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    precision = precision_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    auc = roc_auc_score(y_true, y_score)

    lr_plus = sensitivity / (1 - specificity) if specificity < 1 else np.inf
    lr_minus = (1 - sensitivity) / specificity if specificity > 0 else np.inf
    dor = lr_plus / lr_minus if lr_minus not in (0, np.inf) else np.inf

    return {
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1_score": f1,
        "mcc": mcc,
        "auc": auc,
        "LR+": lr_plus,
        "LR-": lr_minus,
        "DOR": dor,
    }


def print_metrics(title, m):
    print(f"\n--- {title} ---")
    print(f"Confusion matrix: TP={m['TP']} TN={m['TN']} FP={m['FP']} FN={m['FN']}")
    print(f"Accuracy    : {m['accuracy']*100:.2f}%")
    print(f"Precision   : {m['precision']*100:.2f}%")
    print(f"Sensitivity : {m['sensitivity']*100:.2f}%")
    print(f"Specificity : {m['specificity']*100:.2f}%")
    print(f"F1-score    : {m['f1_score']*100:.2f}%")
    print(f"MCC         : {m['mcc']*100:.2f}%")
    print(f"AUC         : {m['auc']*100:.2f}%")
    print(f"LR+         : {m['LR+']:.4f}")
    print(f"LR-         : {m['LR-']:.4f}")
    print(f"DOR         : {m['DOR']:.4f}")

# Final CSVM training + evaluation (train/test split) + plots

In [27]:
def train_and_evaluate_final_model(X_train, y_train, X_test, y_test, best_features, best_hyperparams, out_dir="."):
    X_train_sel, X_test_sel = X_train[:, best_features], X_test[:, best_features]

    clf = SVC(
        C=best_hyperparams["C"], gamma=best_hyperparams["gamma"],
        kernel="rbf", probability=True, random_state=RANDOM_STATE,
    )
    clf.fit(X_train_sel, y_train)

    y_train_pred = clf.predict(X_train_sel)
    y_train_score = clf.predict_proba(X_train_sel)[:, 1]
    y_test_pred = clf.predict(X_test_sel)
    y_test_score = clf.predict_proba(X_test_sel)[:, 1]

    train_metrics = compute_full_metrics(y_train, y_train_pred, y_train_score)
    test_metrics = compute_full_metrics(y_test, y_test_pred, y_test_score)

    # --- Confusion matrix plot (paper Fig. 20) ---
    cm = confusion_matrix(y_test, y_test_pred)
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Normal (0)", "Disease (1)"])
    ax.set_yticklabels(["Normal (0)", "Disease (1)"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title("CSVM Confusion Matrix (test set)")
    fig.colorbar(im)
    fig.tight_layout()
    fig.savefig(f"{out_dir}/confusion_matrix.png", dpi=150)
    plt.close(fig)

    # --- ROC curve plot: train vs test (paper Fig. 22) ---
    fpr_tr, tpr_tr, _ = roc_curve(y_train, y_train_score)
    fpr_te, tpr_te, _ = roc_curve(y_test, y_test_score)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(fpr_tr, tpr_tr, label=f"Train (AUC={train_metrics['auc']:.2f})")
    ax.plot(fpr_te, tpr_te, label=f"Test (AUC={test_metrics['auc']:.2f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("CSVM ROC Curve"); ax.legend()
    fig.tight_layout()
    fig.savefig(f"{out_dir}/roc_curve.png", dpi=150)
    plt.close(fig)

    return clf, train_metrics, test_metrics

Tenfold cross-validation of the final (feature+hyperparameter) model:
K-fold cross-validation

In [28]:
def ten_fold_cross_validation(X, y, best_features, best_hyperparams, out_dir="."):
    X_sel = X[:, best_features]
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

    fold_acc, fold_auc = [], []
    fig, ax = plt.subplots(figsize=(6, 6))

    for fold_i, (train_idx, test_idx) in enumerate(skf.split(X_sel, y), start=1):
        X_tr, X_te = X_sel[train_idx], X_sel[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        scaler = StandardScaler().fit(X_tr)
        X_tr, X_te = scaler.transform(X_tr), scaler.transform(X_te)

        clf = SVC(C=best_hyperparams["C"], gamma=best_hyperparams["gamma"],
                  kernel="rbf", probability=True, random_state=RANDOM_STATE)
        clf.fit(X_tr, y_tr)

        y_pred = clf.predict(X_te)
        y_score = clf.predict_proba(X_te)[:, 1]
        acc = accuracy_score(y_te, y_pred)
        auc = roc_auc_score(y_te, y_score)
        fold_acc.append(acc)
        fold_auc.append(auc)

        fpr, tpr, _ = roc_curve(y_te, y_score)
        ax.plot(fpr, tpr, alpha=0.6, label=f"Fold{fold_i} (AUC={auc:.2f})")

    ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("Tenfold Cross-Validation ROC Curves")
    ax.legend(fontsize=7, loc="lower right")
    fig.tight_layout()
    fig.savefig(f"{out_dir}/ten_fold_roc_curve.png", dpi=150)
    plt.close(fig)

    return {
        "fold_accuracy": fold_acc, "fold_auc": fold_auc,
        "mean_accuracy": float(np.mean(fold_acc)), "mean_auc": float(np.mean(fold_auc)),
    }

# Main Pipeline

In [29]:
def main(heart_disease, target_col="target", out_dir="."):
    X_raw, y, feature_names, corr_with_target = prepare_data(heart_disease, target_col)

    print("=" * 70)
    print("Pearson correlation of each feature with target (paper Fig. 12)")
    print("=" * 70)
    print(corr_with_target.sort_values(ascending=False).round(3))

    # 80:20 split -> 734 train / 184 test, as in the paper
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_raw, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    print(f"\nTrain samples: {len(y_train)} | Test samples: {len(y_test)}")

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    print("\n" + "=" * 70)
    print("STEP 1: Feature Selection using CGA (classical genetic algorithm)")
    print("=" * 70)
    best_features, cga_fitness, cga_history = cga_feature_selection(X_train, y_train)
    selected_names = [f for f, keep in zip(feature_names, best_features) if keep]
    print(f"\nCGA best CV fitness (accuracy): {cga_fitness:.4f}")
    print(f"Selected features ({len(selected_names)}/{len(feature_names)}): {selected_names}")

    print("\n" + "=" * 70)
    print("STEP 2: Hyperparameter Tuning using CPSO (classical PSO)")
    print("=" * 70)
    best_hyperparams, cpso_fitness, cpso_history = cpso_hyperparameter_tuning(
        X_train, y_train, best_features
    )
    print(f"\nCPSO best CV fitness (accuracy): {cpso_fitness:.4f}")
    print(f"Best hyperparameters: C={best_hyperparams['C']:.4f}, gamma={best_hyperparams['gamma']:.4f}")

    print("\n" + "=" * 70)
    print("STEP 3: Final CSVM training & evaluation (80:20 split)")
    print("=" * 70)
    model, train_metrics, test_metrics = train_and_evaluate_final_model(
        X_train, y_train, X_test, y_test, best_features, best_hyperparams, out_dir=out_dir
    )
    print_metrics("Train set", train_metrics)
    print_metrics("Test set", test_metrics)

    print("\n" + "=" * 70)
    print("STEP 4: Tenfold cross-validation of the final model")
    print("=" * 70)
    cv_results = ten_fold_cross_validation(X_raw, y, best_features, best_hyperparams, out_dir=out_dir)
    for i, (a, auc) in enumerate(zip(cv_results["fold_accuracy"], cv_results["fold_auc"]), start=1):
        print(f"Fold {i:2d}: accuracy={a*100:.2f}%  AUC={auc*100:.2f}%")
    print(f"\nMean 10-fold accuracy: {cv_results['mean_accuracy']*100:.2f}%")
    print(f"Mean 10-fold AUC     : {cv_results['mean_auc']*100:.2f}%")

    print(f"\nFigures saved to: {out_dir}/confusion_matrix.png, "
          f"{out_dir}/roc_curve.png, {out_dir}/ten_fold_roc_curve.png")

    return {
        "selected_features": selected_names,
        "best_hyperparams": best_hyperparams,
        "train_metrics": train_metrics,
        "test_metrics": test_metrics,
        "cv_results": cv_results,
    }


if __name__ == "__main__":
    heart_disease = pd.read_csv("UCI-918-11.csv")
    results = main(heart_disease)

Pearson correlation of each feature with target (paper Fig. 12)
ST slope               0.553
exercise angina        0.494
chest pain type        0.471
oldpeak                0.404
sex                    0.305
age                    0.282
fasting blood sugar    0.267
resting bp s           0.108
resting ecg            0.061
cholesterol           -0.233
max heart rate        -0.400
Name: target, dtype: float64

Train samples: 734 | Test samples: 184

STEP 1: Feature Selection using CGA (classical genetic algorithm)
[CGA] Gen 01 | best fitness = 0.8556 | features = 9/11
[CGA] Gen 02 | best fitness = 0.8556 | features = 9/11
[CGA] Gen 03 | best fitness = 0.8556 | features = 9/11
[CGA] Gen 04 | best fitness = 0.8651 | features = 8/11
[CGA] Gen 05 | best fitness = 0.8651 | features = 8/11
[CGA] Gen 06 | best fitness = 0.8651 | features = 8/11
[CGA] Gen 07 | best fitness = 0.8651 | features = 8/11
[CGA] Gen 08 | best fitness = 0.8651 | features = 8/11
[CGA] Gen 09 | best fitness = 0.8651 | fe